# 02 - Dataset2 Linear SVM Model Training

Bu notebook, Dataset2 feature tabloları üzerinden LinearSVC modellerini eğitir veya mevcut model dosyalarını yükler.

Amaç, sonraki değerlendirme aşamalarında kullanılacak model dosyalarını ve temel eğitim özetlerini üretmektir.


## 1. Kütüphaneler

Bu bölümde dosya işlemleri, tablo yönetimi, görselleştirme ve LinearSVC eğitimi için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path, PureWindowsPath
from datetime import datetime
import gc
import json
import math
import os
import time
import traceback
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 2. Ayarlar

Dataset seçimi, LinearSVC parametreleri, debug ayarları ve overwrite davranışı tanımlanır.


In [ ]:
# -----------------------------------------------------------------------------
# Runtime settings
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
CV_FOLDS = 3
SCORING = "f1"

# Colab-first setting requested for Stage 04.
# If Colab RAM crashes, change N_JOBS to 1 or 2 and rerun.
N_JOBS = -1

# Full run by default.
# Set DEBUG_MODE=True only for a small sanity check.
DEBUG_MODE = False
DEBUG_MAX_MODELS = 2

# Optional targeted debug run.
# Disabled by default for the official full run.
DEBUG_FILTER_ENABLED = False
DEBUG_FILTER_ROWS = [
    {
        "dataset_name": "dataset1_varroa",
        "patch_set": "centered80_balanced",
        "feature_set": "hsv_lbp_only",
    },
    {
        "dataset_name": "dataset2_honeybee",
        "patch_set": "centered48_balanced",
        "feature_set": "hsv_lbp_only",
    },
]

# If False, existing model artifacts are skipped and recorded as loaded_existing.
# If True, existing model artifacts are retrained and overwritten.
# Existing model artifacts are kept when OVERWRITE_MODELS=False.
OVERWRITE_MODELS = False

OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

ACTIVE_DATASETS = ["dataset2_honeybee"]

def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()

    start_path = Path(start_path).resolve()

    for candidate in [start_path] + list(start_path.parents):
        has_repo_layout = (candidate / "approaches" / "traditional_ml").exists()
        has_data_dir = (candidate / "data").exists()

        if has_repo_layout and has_data_dir:
            return candidate

    return None


PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise FileNotFoundError("Proje kökü bulunamadı. Notebook'u repo içinde çalıştırın.")

DATASET_CONFIGS = {
    "dataset1_varroa": {
        "dataset_short": "d1",
        "display_name": "Dataset1 Varroa",
        "patch_set_order": [
            "centered80_balanced",
            "centered80_neg4x",
            "centered96_balanced",
            "centered96_neg4x",
            "centered112_balanced",
            "centered112_neg4x",
        ],
    },
    "dataset2_honeybee": {
        "dataset_short": "d2",
        "display_name": "Dataset2 Honeybee",
        "patch_set_order": [
            "centered48_balanced",
            "centered48_neg4x",
            "centered64_balanced",
            "centered64_neg4x",
            "centered80_balanced",
            "centered80_neg4x",
        ],
    },
}

FEATURE_SET_ORDER = [
    "hog8_hsv_lbp",
    "hog8_hsv",
    "hog8_only",
    "hog16_hsv_lbp",
    "hog16_hsv",
    "hog16_only",
    "hsv_lbp_only",
    "hog8_pca_hsv_lbp",
    "hog8_pca_hsv",
    "hog8_pca_only",
    "hog16_pca_hsv_lbp",
    "hog16_pca_hsv",
    "hog16_pca_only",
]

FEATURE_PREFIXES = (
    "hog8_",
    "hog16_",
    "hsv_",
    "lbp_",
)

SPLIT_COLUMN = "split"
LABEL_COLUMN = "patch_label"
PATCH_ID_COLUMN = "patch_id"
TRAIN_SPLIT = "train"
VALID_SPLIT = "valid"
TEST_SPLIT = "test"

EXPECTED_FEATURE_FILES_PER_DATASET = 78

NOTEBOOK_NAME = "02_linear_svm_dataset2_training"
ALGORITHM_NAME = "linear_svm"
ALGORITHM_SHORT = "linsvm"
ALGORITHM_DISPLAY_NAME = "Linear SVM"
ALGORITHM_ORDER_PREFIX = "01"
HYPERPARAMETER_GRID_NAME = "linear_svm_stage04_v1"
PIPELINE_DESCRIPTION = "standard_scaler + linear_svc"
SCALER_USED = True
SCORE_METHOD = "decision_function"

LINEAR_SVM_PARAM_GRID = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
}

PARAM_GRID = LINEAR_SVM_PARAM_GRID


## 3. Yardımcı Fonksiyonlar

Proje kökünü bulma, klasör oluşturma, manifest yollarını okuma ve sayısal değerleri güvenli dönüştürme işlemleri bu bölümde toplanır.


In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        try:
            start_path = Path(__file__).resolve()
        except NameError:
            start_path = Path.cwd().resolve()

    candidates = [Path(start_path)] + list(Path(start_path).parents)
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "approaches").exists():
            return candidate

    configured_root = Path(PROJECT_ROOT)
    if configured_root.exists():
        return configured_root

    raise FileNotFoundError(
        "Proje kökü bulunamadı. Notebook'u proje klasörü içinde çalıştırdığından emin ol."
    )


def timestamp_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def ensure_directories(paths):
    for path in paths:
        Path(path).mkdir(parents=True, exist_ok=True)


def filename_from_manifest_path(path_value):
    if pd.isna(path_value):
        return None
    normalized = str(path_value).replace("\\", "/")
    return normalized.split("/")[-1]


def json_dumps_safe(value):
    try:
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    except TypeError:
        return json.dumps(str(value), ensure_ascii=False)


def safe_float(value):
    try:
        if value is None or pd.isna(value):
            return np.nan
        return float(value)
    except Exception:
        return np.nan


def safe_int(value):
    try:
        if value is None or pd.isna(value):
            return 0
        return int(value)
    except Exception:
        return 0


## 4. Kayıt ve Çıktı Yardımcıları

Tablo ve görsel çıktıları ilgili klasörlere kaydedilir. Mevcut dosya varsa overwrite ayarına göre korunur veya güncellenir.


In [ ]:
def log_output(message):
    print(message)


def log_section(title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def log_dataframe(title, df, max_rows=50):
    print()
    print(title)
    if df is None:
        print("DataFrame yok.")
        return
    display_df = df.head(max_rows).copy()
    print(display_df.to_string(index=False))
    display(display_df)


def log_figure(title, description, figure_path):
    print(f"[FIGURE] {title}: {figure_path}")


def save_dataframe_csv(df, output_path, overwrite=True, index=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        try:
            loaded_df = pd.read_csv(output_path)
            print(f"[LOAD] Existing CSV loaded: {output_path}")
            return loaded_df, "loaded_existing"
        except pd.errors.EmptyDataError:
            print(f"[SKIP] Existing CSV is empty; current DataFrame kept in memory: {output_path}")
            return df, "kept_current_empty_existing"

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing CSV: {output_path}")

    df.to_csv(output_path, index=index, encoding="utf-8-sig")
    print(f"[SAVE] CSV saved: {output_path}")
    return df, "created_or_overwritten"


def save_model_joblib(model, output_path, overwrite=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        loaded_model = joblib.load(output_path)
        print(f"[LOAD] Existing model loaded: {output_path}")
        return loaded_model, "loaded_existing"

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing model: {output_path}")

    joblib.dump(model, output_path)
    print(f"[SAVE] Model saved: {output_path}")
    return model, "created_or_overwritten"


def save_figure(fig, output_path, overwrite=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        print(f"[SKIP] Existing figure kept: {output_path}")
        return output_path, "loaded_existing"

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing figure: {output_path}")

    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"[SAVE] Figure saved: {output_path}")
    return output_path, "created_or_overwritten"


## 5. Dosya Yolları

Feature dosyaları, model klasörü ve bu notebooka ait tablo/görsel çıktı klasörleri tanımlanır.


In [ ]:
PROJECT_ROOT = find_project_root()

STAGE_NAME = "04_linear_svm_model_training"
APPROACH_NAME = "traditional_ml"
APPROACH_DIR = PROJECT_ROOT / "approaches" / APPROACH_NAME
NOTEBOOK_DIR = APPROACH_DIR / "notebooks" / STAGE_NAME
OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

MODELS_ROOT = PROJECT_ROOT / "outputs" / "models"

ensure_directories([TABLES_DIR, FIGURES_DIR, MODELS_ROOT])

log_section("01 LINEAR SVM TRAINING NOTEBOOK STARTED")
log_output(f"PROJECT_ROOT = {PROJECT_ROOT}")
log_output(f"NOTEBOOK_NAME = {NOTEBOOK_NAME}")
log_output(f"ACTIVE_DATASETS = {ACTIVE_DATASETS}")
log_output(f"DEBUG_MODE = {DEBUG_MODE}")
log_output(f"DEBUG_MAX_MODELS = {DEBUG_MAX_MODELS}")
log_output(f"DEBUG_FILTER_ENABLED = {DEBUG_FILTER_ENABLED}")
log_output(f"OVERWRITE_MODELS = {OVERWRITE_MODELS}")
log_output(f"N_JOBS = {N_JOBS}")
log_output(f"MODELS_ROOT = {MODELS_ROOT}")
log_output(f"OUTPUT_DIR = {OUTPUT_DIR}")

## 6. Linear SVM Konfigürasyon Tablosu

Eğitimde kullanılacak sınıflandırıcı ve temel parametreler kısa bir tablo halinde gösterilir.


In [ ]:
log_section("02 LINEAR SVM TRAINING CONFIGURATION")

linear_svm_config_df = pd.DataFrame([
    {"setting": "algorithm_name", "value": ALGORITHM_NAME},
    {"setting": "pipeline", "value": PIPELINE_DESCRIPTION},
    {"setting": "model_type", "value": "LinearSVC"},
    {"setting": "scaler_used", "value": SCALER_USED},
    {"setting": "score_method", "value": SCORE_METHOD},
    {"setting": "class_weight", "value": "None"},
    {"setting": "cv_folds", "value": CV_FOLDS},
    {"setting": "scoring", "value": SCORING},
    {"setting": "n_jobs", "value": N_JOBS},
    {"setting": "hyperparameter_grid_name", "value": HYPERPARAMETER_GRID_NAME},
    {"setting": "output_algorithm_folder", "value": f"{ALGORITHM_ORDER_PREFIX}_{ALGORITHM_NAME}"},
])
log_dataframe("Linear SVM Configuration", linear_svm_config_df)

linear_svm_grid_df = pd.DataFrame([
    {"C": c_value}
    for c_value in LINEAR_SVM_PARAM_GRID["model__C"]
])
log_dataframe("Linear SVM Hyperparameter Grid", linear_svm_grid_df, max_rows=50)

save_dataframe_csv(linear_svm_config_df, TABLES_DIR / "model_training_config.csv", overwrite=OVERWRITE_TABLES, note="Linear SVM config")
save_dataframe_csv(linear_svm_grid_df, TABLES_DIR / "hyperparameter_grid_summary.csv", overwrite=OVERWRITE_TABLES, note="Linear SVM grid summary")


## 7. Eğitim Planı Yardımcıları

Feature manifest dosyaları okunur ve her feature set için model, tablo ve görsel yolları oluşturulur.


In [ ]:
def dataset_output_paths(dataset_name):
    config = DATASET_CONFIGS[dataset_name]
    algorithm_folder = f"{ALGORITHM_ORDER_PREFIX}_{ALGORITHM_NAME}"
    base_dir = MODELS_ROOT / dataset_name / algorithm_folder
    return {
        "base_dir": base_dir,
        "models_dir": base_dir,
        "tables_dir": base_dir / "tables",
        "figures_dir": base_dir / "figures",
    }


def load_feature_manifest(dataset_name):
    features_dir = PROJECT_ROOT / "data" / "features" / dataset_name
    manifest_path = features_dir / "feature_manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Feature manifest not found: {manifest_path}")
    manifest_df = pd.read_csv(manifest_path)
    manifest_df["manifest_path"] = relative_path(manifest_path)
    manifest_df["features_dir"] = relative_path(features_dir)
    return manifest_df


def prepare_training_plan_for_dataset(dataset_name):
    config = DATASET_CONFIGS[dataset_name]
    dataset_short = config["dataset_short"]
    features_dir = PROJECT_ROOT / "data" / "features" / dataset_name
    manifest_df = load_feature_manifest(dataset_name).copy()

    manifest_df["feature_file_name"] = manifest_df["feature_file_path"].apply(filename_from_manifest_path)
    manifest_df["resolved_feature_file_path"] = manifest_df["feature_file_name"].apply(
        lambda name: relative_path(features_dir / name)
    )
    manifest_df["feature_file_exists"] = manifest_df["resolved_feature_file_path"].apply(lambda p: Path(p).exists())

    patch_order_map = {name: idx for idx, name in enumerate(config["patch_set_order"])}
    feature_order_map = {name: idx for idx, name in enumerate(FEATURE_SET_ORDER)}

    manifest_df["patch_set_order"] = manifest_df["patch_set"].map(patch_order_map).fillna(9999).astype(int)
    manifest_df["feature_set_order"] = manifest_df["feature_set"].map(feature_order_map).fillna(9999).astype(int)
    manifest_df = manifest_df.sort_values(["patch_set_order", "feature_set_order", "patch_set", "feature_set"]).reset_index(drop=True)

    paths = dataset_output_paths(dataset_name)
    ensure_directories(paths.values())

    model_ids = []
    model_paths = []
    for idx in range(len(manifest_df)):
        model_id = f"{dataset_short}_{ALGORITHM_SHORT}_{idx + 1:03d}"
        safe_patch_set = str(manifest_df.loc[idx, "patch_set"])
        safe_feature_set = str(manifest_df.loc[idx, "feature_set"])
        model_filename = f"{model_id}__{safe_patch_set}__{safe_feature_set}.joblib"
        model_ids.append(model_id)
        model_paths.append(relative_path(paths["models_dir"] / model_filename))

    manifest_df["model_index"] = np.arange(1, len(manifest_df) + 1)
    manifest_df["model_id"] = model_ids
    manifest_df["model_file_path"] = model_paths
    manifest_df["algorithm_name"] = ALGORITHM_NAME
    manifest_df["algorithm_short"] = ALGORITHM_SHORT
    manifest_df["dataset_short"] = dataset_short
    manifest_df["score_method"] = SCORE_METHOD
    manifest_df["scaler_used"] = SCALER_USED
    manifest_df["cv_folds"] = CV_FOLDS
    manifest_df["scoring"] = SCORING
    manifest_df["hyperparameter_grid_name"] = HYPERPARAMETER_GRID_NAME
    manifest_df["model_fit_split"] = TRAIN_SPLIT
    manifest_df["valid_used_for_fit_or_selection"] = False
    manifest_df["test_used_for_fit_or_selection"] = False

    return manifest_df



def create_combined_training_plan():
    plans = []
    for dataset_name in ACTIVE_DATASETS:
        plan = prepare_training_plan_for_dataset(dataset_name)
        plans.append(plan)

    combined = pd.concat(plans, ignore_index=True)

    combined = combined.sort_values(
        ["dataset_name", "model_index"]
    ).reset_index(drop=True)

    if DEBUG_MODE and DEBUG_FILTER_ENABLED:
        selected_parts = []

        for filter_row in DEBUG_FILTER_ROWS:
            filtered = combined[
                (combined["dataset_name"] == filter_row["dataset_name"])
                & (combined["patch_set"] == filter_row["patch_set"])
                & (combined["feature_set"] == filter_row["feature_set"])
            ].copy()

            if len(filtered) == 0:
                raise ValueError(
                    "DEBUG filter did not match any planned model. "
                    f"Filter was: {filter_row}"
                )

            selected_parts.append(filtered.head(1))

        combined = pd.concat(selected_parts, ignore_index=True)
        combined = combined.head(DEBUG_MAX_MODELS).reset_index(drop=True)

    elif DEBUG_MODE:
        combined = combined.head(DEBUG_MAX_MODELS).reset_index(drop=True)

    return combined


def select_feature_columns(df):
    feature_columns = [col for col in df.columns if str(col).startswith(FEATURE_PREFIXES)]
    return feature_columns


def validate_feature_dataframe(df, plan_row):
    errors = []
    if SPLIT_COLUMN not in df.columns:
        errors.append(f"Missing split column: {SPLIT_COLUMN}")
    if LABEL_COLUMN not in df.columns:
        errors.append(f"Missing label column: {LABEL_COLUMN}")

    feature_columns = select_feature_columns(df)
    if len(feature_columns) == 0:
        errors.append("No feature columns found using FEATURE_PREFIXES.")

    if SPLIT_COLUMN in df.columns:
        missing_splits = sorted(set([TRAIN_SPLIT, VALID_SPLIT, TEST_SPLIT]) - set(df[SPLIT_COLUMN].astype(str).unique()))
        if missing_splits:
            errors.append(f"Missing expected split(s): {missing_splits}")

    if LABEL_COLUMN in df.columns:
        labels = sorted(df[LABEL_COLUMN].dropna().unique().tolist())
        if labels != [0, 1]:
            errors.append(f"Unexpected labels: {labels}. Expected [0, 1].")

    if feature_columns:
        numeric_check = all(pd.api.types.is_numeric_dtype(df[col]) for col in feature_columns)
        if not numeric_check:
            errors.append("At least one selected feature column is not numeric.")

    return errors, feature_columns


def split_train_xy(df, feature_columns):
    train_df = df[df[SPLIT_COLUMN].astype(str) == TRAIN_SPLIT].copy()
    X_train = train_df[feature_columns].to_numpy(dtype=np.float32, copy=True)
    y_train = train_df[LABEL_COLUMN].to_numpy(dtype=np.int32, copy=True)
    return X_train, y_train


def summarize_split_label_counts(df):
    if SPLIT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
        return pd.DataFrame()
    summary = (
        df.groupby([SPLIT_COLUMN, LABEL_COLUMN])
        .size()
        .reset_index(name="count")
        .sort_values([SPLIT_COLUMN, LABEL_COLUMN])
    )
    return summary


## 8. Eğitim Planı

Seçilen dataset için kullanılabilir feature setleri listelenir ve eğitim planı hazırlanır.


In [ ]:

log_section("03 BUILD TRAINING PLAN")
training_plan_df = create_combined_training_plan()

log_output(f"Training plan row count: {len(training_plan_df)}")
log_dataframe("Training Plan Preview", training_plan_df[[
    "dataset_name", "model_id", "patch_set", "feature_set", "feature_column_count", "resolved_feature_file_path", "feature_file_exists"
]], max_rows=20)

feature_availability_df = training_plan_df[[
    "dataset_name", "model_id", "patch_set", "feature_set", "resolved_feature_file_path", "feature_file_exists", "status", "nan_count", "inf_count"
]].copy()
log_dataframe("Feature File Availability Check", feature_availability_df, max_rows=20)

manifest_audit_df = training_plan_df.groupby("dataset_name").agg(
    planned_model_count=("model_id", "count"),
    available_feature_file_count=("feature_file_exists", "sum"),
    min_feature_count=("feature_column_count", "min"),
    max_feature_count=("feature_column_count", "max"),
    total_rows=("row_count", "sum"),
).reset_index()
log_dataframe("Manifest Audit Summary", manifest_audit_df)

save_dataframe_csv(feature_availability_df, TABLES_DIR / "feature_file_availability_check.csv", overwrite=OVERWRITE_TABLES, note="Feature availability")
save_dataframe_csv(manifest_audit_df, TABLES_DIR / "manifest_audit_summary.csv", overwrite=OVERWRITE_TABLES, note="Manifest audit")


## 9. Feature Şema Kontrolü

Eğitimden önce feature dosyalarının varlığı, satır sayıları ve temel kolon bilgileri kontrol edilir.


In [ ]:
log_section("04 FEATURE SCHEMA AUDIT")

schema_records = []

for _, row in training_plan_df.iterrows():
    feature_path = Path(row["resolved_feature_file_path"])

    record = {
        "model_id": row["model_id"],
        "dataset_name": row["dataset_name"],
        "patch_set": row["patch_set"],
        "feature_set": row["feature_set"],
        "feature_file_path": relative_path(feature_path),
        "feature_file_exists": feature_path.exists(),
        "row_count_manifest": safe_int(row.get("row_count")),
        "train_row_count_manifest": safe_int(row.get("train_row_count")),
        "valid_row_count_manifest": safe_int(row.get("valid_row_count")),
        "test_row_count_manifest": safe_int(row.get("test_row_count")),
        "metadata_column_count_manifest": safe_int(row.get("metadata_column_count")),
        "feature_column_count_manifest": safe_int(row.get("feature_column_count")),
        "total_column_count_manifest": safe_int(row.get("total_column_count")),
        "nan_count_manifest": safe_int(row.get("nan_count")),
        "inf_count_manifest": safe_int(row.get("inf_count")),
        "status_manifest": row.get("status"),
    }

    errors = []

    if not feature_path.exists():
        errors.append("feature_file_missing")

    if safe_int(row.get("feature_column_count")) <= 0:
        errors.append("feature_column_count_not_positive")

    if safe_int(row.get("nan_count")) != 0:
        errors.append("manifest_nan_count_nonzero")

    if safe_int(row.get("inf_count")) != 0:
        errors.append("manifest_inf_count_nonzero")

    record["validation_errors"] = "; ".join(errors)
    schema_records.append(record)

feature_schema_audit_df = pd.DataFrame(schema_records)

log_dataframe("Feature Schema Audit", feature_schema_audit_df, max_rows=20)
save_dataframe_csv(
    feature_schema_audit_df,
    TABLES_DIR / "feature_schema_audit.csv",
    overwrite=OVERWRITE_TABLES,
    note="Manifest-based feature schema audit"
)


## 10. LinearSVC Pipeline

StandardScaler ve LinearSVC içeren eğitim pipeline tanımlanır.


In [ ]:
def build_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearSVC(
            class_weight=None,
            random_state=RANDOM_STATE,
            dual=False,
            max_iter=20000,
        )),
    ])

log_section("05 PIPELINE READY")
log_output("Linear SVM pipeline: StandardScaler -> LinearSVC(class_weight=None, dual=False, max_iter=20000)")


## 11. Model Eğitimi veya Mevcut Modeli Yükleme

Her feature set için model dosyası kontrol edilir. Model mevcutsa ve overwrite kapalıysa dosya korunur; aksi durumda model yeniden eğitilir.


In [ ]:
def train_single_model(plan_row):
    model_id = plan_row["model_id"]
    dataset_name = plan_row["dataset_name"]
    feature_path = Path(plan_row["resolved_feature_file_path"])
    model_path = Path(plan_row["model_file_path"])

    start_time = time.time()

    registry_base = {
        "model_id": model_id,
        "dataset_name": dataset_name,
        "dataset_short": plan_row.get("dataset_short"),
        "algorithm_name": ALGORITHM_NAME,
        "algorithm_short": ALGORITHM_SHORT,
        "model_index": int(plan_row["model_index"]),
        "patch_set": plan_row.get("patch_set"),
        "patch_size": plan_row.get("patch_size"),
        "ratio_variant": plan_row.get("ratio_variant"),
        "feature_set": plan_row.get("feature_set"),
        "feature_file_path": relative_path(feature_path),
        "model_file_path": relative_path(model_path),
        "pipeline_steps": PIPELINE_DESCRIPTION,
        "model_type": "LinearSVC",
        "scaler_used": SCALER_USED,
        "model_fit_split": TRAIN_SPLIT,
        "valid_used_for_fit_or_selection": False,
        "test_used_for_fit_or_selection": False,
        "class_weight": "None",
        "hyperparameter_grid_name": HYPERPARAMETER_GRID_NAME,
        "search_strategy": "train_split_gridsearch_cv",
        "cv_used": True,
        "cv_folds": CV_FOLDS,
        "scoring": SCORING,
        "score_method": SCORE_METHOD,
        "random_state": RANDOM_STATE,
        "created_at": timestamp_now(),
    }

    if model_path.exists() and not OVERWRITE_MODELS:
        runtime = time.time() - start_time
        return {
            **registry_base,
            "fit_status": "loaded_existing",
            "fit_runtime_seconds": runtime,
            "selected_params_json": None,
            "best_cv_score": np.nan,
            "feature_count_actual": np.nan,
            "train_row_count_actual": np.nan,
            "train_positive_count_actual": np.nan,
            "train_negative_count_actual": np.nan,
            "error_message": "",
            "notes": "Existing model artifact found and OVERWRITE_MODELS=False.",
        }

    if not feature_path.exists():
        runtime = time.time() - start_time
        return {
            **registry_base,
            "fit_status": "failed",
            "fit_runtime_seconds": runtime,
            "selected_params_json": None,
            "best_cv_score": np.nan,
            "feature_count_actual": np.nan,
            "train_row_count_actual": np.nan,
            "train_positive_count_actual": np.nan,
            "train_negative_count_actual": np.nan,
            "error_message": f"Feature file not found: {feature_path}",
            "notes": "Feature file missing.",
        }

    try:
        df = pd.read_parquet(feature_path)
        validation_errors, feature_columns = validate_feature_dataframe(df, plan_row)
        if validation_errors:
            raise ValueError("; ".join(validation_errors))

        X_train, y_train = split_train_xy(df, feature_columns)
        if len(np.unique(y_train)) < 2:
            raise ValueError("Train split has fewer than two classes.")

        pipeline = build_pipeline()
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=PARAM_GRID,
            scoring=SCORING,
            cv=cv,
            n_jobs=N_JOBS,
            refit=True,
            verbose=0,
            error_score="raise",
            return_train_score=False,
        )
        grid_search.fit(X_train, y_train)

        save_action = save_model_joblib(
            grid_search.best_estimator_,
            model_path,
            overwrite=OVERWRITE_MODELS,
            note=f"{ALGORITHM_NAME} model artifact: {model_id}",
        )

        runtime = time.time() - start_time
        registry_row = {
            **registry_base,
            "fit_status": "trained" if save_action in ["created", "overwritten"] else save_action,
            "fit_runtime_seconds": runtime,
            "selected_params_json": json_dumps_safe(grid_search.best_params_),
            "best_cv_score": safe_float(grid_search.best_score_),
            "feature_count_actual": len(feature_columns),
            "train_row_count_actual": len(y_train),
            "train_positive_count_actual": int(np.sum(y_train == 1)),
            "train_negative_count_actual": int(np.sum(y_train == 0)),
            "error_message": "",
            "notes": "Trained LinearSVC with train split GridSearchCV only.",
        }

        del df, X_train, y_train, grid_search, pipeline
        gc.collect()
        return registry_row

    except Exception as exc:
        runtime = time.time() - start_time
        error_text = f"{type(exc).__name__}: {exc}"
        traceback_text = traceback.format_exc(limit=3)
        log_output(f"[FAILED] {model_id}: {error_text}\n{traceback_text}")
        gc.collect()
        return {
            **registry_base,
            "fit_status": "failed",
            "fit_runtime_seconds": runtime,
            "selected_params_json": None,
            "best_cv_score": np.nan,
            "feature_count_actual": np.nan,
            "train_row_count_actual": np.nan,
            "train_positive_count_actual": np.nan,
            "train_negative_count_actual": np.nan,
            "error_message": error_text,
            "notes": traceback_text,
        }


log_section("06 MODEL TRAINING LOOP")
registry_rows = []
for row_number, (_, plan_row) in enumerate(training_plan_df.iterrows(), start=1):
    log_output(
        f"[{row_number}/{len(training_plan_df)}] Training or loading {plan_row['model_id']} | "
        f"{plan_row['dataset_name']} | {plan_row['patch_set']} | {plan_row['feature_set']}"
    )
    registry_row = train_single_model(plan_row)
    registry_rows.append(registry_row)

model_registry_df = pd.DataFrame(registry_rows)
log_dataframe("Model Registry Preview", model_registry_df, max_rows=20)


## 12. Registry ve Durum Tabloları

Model kayıt tablosu, eğitim durumu ve kısa özet tabloları kaydedilir.


In [ ]:

log_section("07 SAVE REGISTRY AND STATUS TABLES")

status_counts_df = (
    model_registry_df.groupby(["dataset_name", "algorithm_name", "fit_status"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "algorithm_name", "fit_status"])
)
log_dataframe("Training Status Counts", status_counts_df)

expected_by_dataset = training_plan_df.groupby("dataset_name").size().reset_index(name="expected_model_count")
actual_status = model_registry_df.pivot_table(
    index="dataset_name",
    columns="fit_status",
    values="model_id",
    aggfunc="count",
    fill_value=0,
).reset_index()
training_status_df = expected_by_dataset.merge(actual_status, on="dataset_name", how="left")
for col in ["trained", "loaded_existing", "failed"]:
    if col not in training_status_df.columns:
        training_status_df[col] = 0
training_status_df["completed_or_loaded_count"] = training_status_df["trained"] + training_status_df["loaded_existing"]
training_status_df["missing_or_failed_count"] = training_status_df["expected_model_count"] - training_status_df["completed_or_loaded_count"]
training_status_df["debug_mode"] = DEBUG_MODE
training_status_df["status"] = np.where(
    DEBUG_MODE,
    "DEBUG_RUN_COMPLETED",
    np.where(training_status_df["missing_or_failed_count"] == 0, "READY_FOR_NEXT_STAGE", "COMPLETED_WITH_WARNINGS"),
)
log_dataframe("Training Status Summary", training_status_df)

failed_models_df = model_registry_df[model_registry_df["fit_status"] == "failed"].copy()
if len(failed_models_df) > 0:
    log_dataframe("Failed Models", failed_models_df[["model_id", "dataset_name", "patch_set", "feature_set", "error_message"]], max_rows=50)
else:
    log_output("No failed models recorded.")

# Save combined stage-level tables.
save_dataframe_csv(training_plan_df, TABLES_DIR / "training_plan.csv", overwrite=OVERWRITE_TABLES, note="Combined Stage 04 training plan")
save_dataframe_csv(model_registry_df, TABLES_DIR / "model_registry.csv", overwrite=OVERWRITE_TABLES, note="Combined Stage 04 model registry")
save_dataframe_csv(training_status_df, TABLES_DIR / "training_status.csv", overwrite=OVERWRITE_TABLES, note="Combined Stage 04 training status")
save_dataframe_csv(failed_models_df, TABLES_DIR / "failed_models.csv", overwrite=OVERWRITE_TABLES, note="Failed model records")
save_dataframe_csv(status_counts_df, TABLES_DIR / "training_status_counts.csv", overwrite=OVERWRITE_TABLES, note="Training status counts")

# Dataset ve algoritma klasörüne ait tabloları kaydet.
for dataset_name in ACTIVE_DATASETS:
    paths = dataset_output_paths(dataset_name)
    dataset_registry = model_registry_df[model_registry_df["dataset_name"] == dataset_name].copy()
    dataset_status = training_status_df[training_status_df["dataset_name"] == dataset_name].copy()
    save_dataframe_csv(dataset_registry, paths["tables_dir"] / "model_registry.csv", overwrite=OVERWRITE_TABLES, note=f"{dataset_name} algorithm model registry")
    save_dataframe_csv(dataset_status, paths["tables_dir"] / "training_status.csv", overwrite=OVERWRITE_TABLES, note=f"{dataset_name} algorithm training status")

notebook_status = "DEBUG_RUN_COMPLETED" if DEBUG_MODE else (
    "READY_FOR_NEXT_STAGE" if (training_status_df["missing_or_failed_count"] == 0).all() else "COMPLETED_WITH_WARNINGS"
)
notebook_status_df = pd.DataFrame([{
    "notebook_name": NOTEBOOK_NAME,
    "algorithm_name": ALGORITHM_NAME,
    "active_datasets": ",".join(ACTIVE_DATASETS),
    "debug_mode": DEBUG_MODE,
    "overwrite_models": OVERWRITE_MODELS,
    "expected_model_count": len(training_plan_df),
    "trained_count": int((model_registry_df["fit_status"] == "trained").sum()),
    "loaded_existing_count": int((model_registry_df["fit_status"] == "loaded_existing").sum()),
    "failed_count": int((model_registry_df["fit_status"] == "failed").sum()),
    "status": notebook_status,
    "created_at": timestamp_now(),
}])
log_dataframe("Notebook Status", notebook_status_df)
save_dataframe_csv(notebook_status_df, TABLES_DIR / "notebook_status.csv", overwrite=OVERWRITE_TABLES, note="Notebook final status")


## 13. Özet Görseller

Eğitim süresi ve feature set boyutlarını karşılaştıran görseller oluşturulur.


In [ ]:

log_section("08 SUMMARY FIGURES")

if len(status_counts_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_df = status_counts_df.copy()
    labels = plot_df["dataset_name"] + "\n" + plot_df["fit_status"]
    ax.bar(labels, plot_df["count"])
    ax.set_title(f"{ALGORITHM_DISPLAY_NAME} Training Status Counts")
    ax.set_ylabel("Model count")
    ax.tick_params(axis="x", rotation=45)
    fig_path = FIGURES_DIR / "training_status_counts.png"
    save_figure(fig, fig_path, overwrite=OVERWRITE_FIGURES, note="Training status counts figure")
    plt.close(fig)
    log_figure("Training Status Counts", "Model artifact status counts by dataset.", fig_path)

if len(training_plan_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    dim_df = training_plan_df.copy()
    dim_df["feature_label"] = dim_df["dataset_short"] + "_" + dim_df["model_index"].astype(str).str.zfill(3)
    ax.plot(np.arange(len(dim_df)), dim_df["feature_column_count"].astype(float).to_numpy(), marker="o", linewidth=1)
    ax.set_title(f"{ALGORITHM_DISPLAY_NAME} Feature Dimension by Planned Model")
    ax.set_xlabel("Planned model order")
    ax.set_ylabel("Feature column count")
    fig_path = FIGURES_DIR / "feature_dimension_by_model.png"
    save_figure(fig, fig_path, overwrite=OVERWRITE_FIGURES, note="Feature dimension by model figure")
    plt.close(fig)
    log_figure("Feature Dimension by Model", "Feature vector size for each planned model.", fig_path)


## 14. Final Durum

Notebook sonunda eğitim durumu kısa olarak özetlenir.


In [ ]:
log_section("09 FINAL DURUM")

summary_lines = [
    f"Notebook: {NOTEBOOK_NAME}",
    f"Algorithm: {ALGORITHM_NAME}",
    f"Active datasets: {', '.join(ACTIVE_DATASETS)}",
    f"Debug mode: {DEBUG_MODE}",
    f"Expected model count in this run: {len(training_plan_df)}",
    f"Trained model count: {int((model_registry_df['fit_status'] == 'trained').sum())}",
    f"Loaded existing model count: {int((model_registry_df['fit_status'] == 'loaded_existing').sum())}",
    f"Failed model count: {int((model_registry_df['fit_status'] == 'failed').sum())}",
    f"Notebook status: {notebook_status}",
]

for line in summary_lines:
    log_output(line)

log_output("Linear SVM model training notebook tamamlandı.")
